In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

CSV_PATH = "cleaned_trade_master.csv"
YEARS = [2023, 2024, 2025, 2026, 2027, 2028]

# -----------------------------
# 1) Load, de-duplicate columns, prep
# -----------------------------
df = pd.read_csv(CSV_PATH)

# DROP duplicate columns (keep the first occurrence)
df = df.loc[:, ~df.columns.duplicated()].copy()

# Ensure target present & drop NaNs in target
df = df.dropna(subset=["Value_2023"]).copy()

# We’re focusing on countries (ignore product granularity).
# Keep only the columns we want IF they exist.
desired_num = [
    "Trade_Balance_2023","Growth_2019_2023","Growth_2022_2023",
    "World_Growth","World_Import_Rank","Avg_Distance_km",
    "Concentration","World_Import_Share","Avg_Tariff"
]
desired_cat = ["Country","Direction"]

# Intersect with actual columns to avoid any KeyError
num_cols = [c for c in desired_num if c in df.columns]
cat_cols = [c for c in desired_cat if c in df.columns]

# Features / target
X = df[num_cols + cat_cols].copy()
y = df["Value_2023"].astype(float)

# -----------------------------
# 2) Preprocessing & split
# -----------------------------
preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline(steps=[
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# 3) Models
# -----------------------------
lr = Pipeline(steps=[("prep", preprocess), ("m", LinearRegression())])
xgb = Pipeline(steps=[("prep", preprocess), ("m", XGBRegressor(
    n_estimators=400, learning_rate=0.07, max_depth=6,
    subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=2
))])

lr.fit(X_train, y_train)
xgb.fit(X_train, y_train)

def eval_model(name, model, X_te, y_te):
    pred = model.predict(X_te)
    r2   = r2_score(y_te, pred)
    rmse = mean_squared_error(y_te, pred, squared=False)
    mae  = mean_absolute_error(y_te, pred)
    print(f"\n{name}")
    print(f"R²   : {r2:.4f}")
    print(f"RMSE : {rmse:,.0f}")
    print(f"MAE  : {mae:,.0f}")
    return pred

_ = eval_model("Linear Regression", lr,  X_test, y_test)
_ = eval_model("XGBoost Regressor", xgb, X_test, y_test)

# -----------------------------
# 4) Row-level base predictions (2023)
# -----------------------------
df["Pred_2023_LR"]  = lr.predict(X)
df["Pred_2023_XGB"] = xgb.predict(X)

# -----------------------------
# 5) Simple forecast 2024–2028 by compounding World_Growth
# -----------------------------
g = (df["World_Growth"].astype(float).fillna(0.0) / 100.0) if "World_Growth" in df.columns else pd.Series(0.0, index=df.index)

def compound(base_series, growth_series, out_years):
    out = {}
    val = base_series.values.astype(float).copy()
    out[2023] = val.copy()
    for y in out_years[1:]:
        val = val * (1.0 + growth_series.values)
        out[y] = val.copy()
    return out

lr_track  = compound(df["Pred_2023_LR"],  g, YEARS)
xgb_track = compound(df["Pred_2023_XGB"], g, YEARS)

# -----------------------------
# 6) Aggregate to COUNTRY level (totals and averages)
# -----------------------------
def to_country_table(model_name, track_dict):
    frames = []
    for year, vals in track_dict.items():
        tmp = pd.DataFrame({
            "Country": df["Country"].values if "Country" in df.columns else "ALL",
            "Year": year,
            "Forecast_Value": vals
        })
        totals = tmp.groupby(["Country","Year"], as_index=False)["Forecast_Value"].sum()
        totals["Metric"] = "Total"
        avgs   = tmp.groupby(["Country","Year"], as_index=False)["Forecast_Value"].mean()
        avgs["Metric"] = "Average"
        out = pd.concat([totals, avgs], ignore_index=True)
        out["Model"] = model_name
        frames.append(out)
    return pd.concat(frames, ignore_index=True)

country_lr  = to_country_table("Linear",  lr_track)
country_xgb = to_country_table("XGBoost", xgb_track)

country_all = pd.concat([country_lr, country_xgb], ignore_index=True)
country_all.to_csv("country_forecast_totals_averages_2023_2028.csv", index=False)
print("\nSaved: country_forecast_totals_averages_2023_2028.csv")

# -----------------------------
# 7) Quick visuals
# -----------------------------
# a) Top 6 countries by 2023 XGB Total
all_countries = country_all["Country"].unique().tolist()

plot_df = country_all[
    (country_all["Model"]=="XGBoost") &
    (country_all["Metric"]=="Total") &
    (country_all["Country"].isin(all_countries))
].pivot_table(index="Year", columns="Country", values="Forecast_Value", aggfunc="sum").sort_index()

plt.figure(figsize=(12,7))
fig.patch.set_facecolor("whitesmoke")   # outer figure
ax.set_facecolor("#f9f9f9") 
for c in plot_df.columns:
    plt.plot(plot_df.index, plot_df[c], marker="o", label=c)

plt.title("All 15 Countries — Total Forecast (XGBoost)")
plt.xlabel("Year")
plt.ylabel("Forecasted Total Trade Value")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")  # legend outside chart
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("all_countries_total_xgb.png", dpi=150)
plt.show()

# b) Average country trend — Linear vs XGB
avg_trend = (country_all[country_all["Metric"]=="Average"]
             .groupby(["Model","Year"], as_index=False)["Forecast_Value"].mean())

plt.figure(figsize=(8,5))
for name, grp in avg_trend.groupby("Model"):
    plt.plot(grp["Year"], grp["Forecast_Value"], marker="o", label=name)
plt.title("Average Country Forecast — Linear vs XGBoost")
plt.xlabel("Year"); plt.ylabel("Average Predicted Trade Value")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig("avg_country_forecast_comparison.png", dpi=140)
plt.show()



Linear Regression
R²   : 0.1438
RMSE : 3,991,388
MAE  : 997,276

XGBoost Regressor
R²   : 0.8249
RMSE : 1,804,910
MAE  : 260,198

Saved: country_forecast_totals_averages_2023_2028.csv


NameError: name 'fig' is not defined

<Figure size 1200x700 with 0 Axes>